In [96]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input

In [97]:
df = pd.read_csv(r"C:\Users\kuash\Downloads\archive\Gold Price.csv")

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

data = df[['Price']].values

In [98]:
split_raw = int(len(data) * 0.8)

train_data = data[:split_raw]
test_data = data[split_raw - 60:]

In [99]:
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_data)
test_scaled = scaler.transform(test_data)

In [100]:
def create_sequences(data, time_step=60):
    X, y = [], []
    for i in range(time_step, len(data)):
        X.append(data[i-time_step:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_scaled, 60)
X_test, y_test = create_sequences(test_scaled, 60)

X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

In [101]:
model = Sequential()
model.add(Input(shape=(60, 1)))
model.add(LSTM(50, return_sequences=True))
model.add(LSTM(50))
model.add(Dense(1))

model.compile(optimizer='adam', loss='mse')

model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    verbose=1
)

Epoch 1/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - loss: 0.0117
Epoch 2/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - loss: 4.6709e-04
Epoch 3/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - loss: 4.8288e-04
Epoch 4/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - loss: 4.8558e-04
Epoch 5/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - loss: 4.2551e-04
Epoch 6/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 4.0702e-04
Epoch 7/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - loss: 4.2652e-04
Epoch 8/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - loss: 3.9646e-04
Epoch 9/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 3.8507e-04
Epoch 10/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - loss: 3.6630e-04
Epoch 11/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - loss: 3.3690e-04
Epoch 12/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 3.3532e-04
Epoch 13/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 3.6153e-04
Epoch 14/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - loss: 2.9891e-04
Epoch 15/20
76/76 ━

In [93]:
pred = model.predict(X_test)
y_pred_inv = scaler.inverse_transform(pred)
y_test_inv = scaler.inverse_transform(y_test.reshape(-1, 1))

20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step


In [94]:
rmse = np.sqrt(mean_squared_error(y_test_inv, y_pred_inv))
mae = mean_absolute_error(y_test_inv, y_pred_inv)
r2 = r2_score(y_test_inv, y_pred_inv)
mape = np.mean(np.abs((y_test_inv - y_pred_inv) / y_test_inv)) * 100

print("\n=== LSTM MODEL RESULTS ===")
print(f"MAE: {mae:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"R2: {r2:.6f}")
print(f"MAPE: {mape:.6f}")


=== LSTM MODEL RESULTS ===
MAE: 2707.011090
RMSE: 4375.816823
R2: 0.954886
MAPE: 2.692631


In [95]:
import os

os.makedirs("models", exist_ok=True)

In [92]:
model.save("models/lstm_model.keras")